# Import e test del modello di regressione logistica per il mutuo

Questo notebook:

- importa il modello esportato in `modello_mutuo.joblib`
- ispeziona la struttura del pipeline
- mostra le feature originali attese dal modello
- mostra le categorie accettate per le feature categoriche
- costruisce dati giocattolo **coerenti con il formato atteso**
- ottiene previsioni sia favorevoli sia sfavorevoli


In [1]:
import pandas as pd
import numpy as np
from joblib import load
from pprint import pprint


## 1. Caricamento del modello

Assunzione: il file `modello_mutuo.joblib` si trova nella stessa cartella del notebook.


In [2]:
model = load("modello_mutuo.joblib")
model


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Dependents',
                                                   'ApplicantIncome',
                                                   'CoapplicantIncome',
                                                   'LoanAmount',
                                                   'Loan_Amount_Term',
                                                   'Credit_History']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Gender', 'Married',
                                                   'Education', 'Self_Employed',
                                                   'Property_Area'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

## 2. Ispezione della struttura del pipeline


In [3]:
print("Passi del pipeline principale:")
for nome, step in model.named_steps.items():
    print(f"- {nome}: {type(step).__name__}")


Passi del pipeline principale:
- preprocessor: ColumnTransformer
- classifier: LogisticRegression


In [4]:
preprocessor = model.named_steps['preprocessor']
classifier = model.named_steps['classifier']

print("Preprocessor:", type(preprocessor).__name__)
print("Classifier:", type(classifier).__name__)


Preprocessor: ColumnTransformer
Classifier: LogisticRegression


## 3. Feature originali attese dal modello

Queste sono le colonne che il pipeline si aspetta **prima** della trasformazione one-hot/scaling.


In [5]:
feature_originali = list(model.feature_names_in_)
feature_originali


['Gender',
 'Married',
 'Dependents',
 'Education',
 'Self_Employed',
 'ApplicantIncome',
 'CoapplicantIncome',
 'LoanAmount',
 'Loan_Amount_Term',
 'Credit_History',
 'Property_Area']

In [6]:
for i, f in enumerate(feature_originali, start=1):
    print(f"{i}. {f}")


1. Gender
2. Married
3. Dependents
4. Education
5. Self_Employed
6. ApplicantIncome
7. CoapplicantIncome
8. LoanAmount
9. Loan_Amount_Term
10. Credit_History
11. Property_Area


## 4. Feature numeriche e categoriche

Recuperiamo le colonne gestite dai due rami del `ColumnTransformer`.


In [7]:
transformers = preprocessor.transformers_
for nome, trasf, cols in transformers:
    print(f"\nTrasformatore: {nome}")
    print("Tipo:", type(trasf).__name__)
    print("Colonne:", list(cols))



Trasformatore: num
Tipo: Pipeline
Colonne: ['Dependents', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']

Trasformatore: cat
Tipo: Pipeline
Colonne: ['Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area']


In [8]:
numeric_features = list(preprocessor.transformers_[0][2])
categorical_features = list(preprocessor.transformers_[1][2])

print("Feature numeriche:")
print(numeric_features)

print("\nFeature categoriche:")
print(categorical_features)


Feature numeriche:
['Dependents', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']

Feature categoriche:
['Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area']


## 5. Valori accettati dal modello

Per le variabili categoriche possiamo ispezionare direttamente le categorie apprese dal `OneHotEncoder`.

Per le variabili numeriche **non esiste un insieme chiuso di valori ammessi**: il modello accetta numeri coerenti con il significato della colonna.


In [9]:
cat_pipeline = preprocessor.named_transformers_['cat']
onehot = cat_pipeline.named_steps['onehot']

categorie_accettate = {
    colonna: list(categorie)
    for colonna, categorie in zip(categorical_features, onehot.categories_)
}

categorie_accettate


{'Gender': ['Female', 'Male'],
 'Married': ['No', 'Yes'],
 'Education': ['Graduate', 'Not Graduate'],
 'Self_Employed': ['No', 'Yes'],
 'Property_Area': ['Rural', 'Semiurban', 'Urban']}

In [11]:
print("Categorie accettate per le feature categoriche:
")
for colonna, categorie in categorie_accettate.items():
    print(f"- {colonna}: {categorie}")


SyntaxError: unterminated string literal (detected at line 1) (1902003538.py, line 1)

In [12]:
print("Per le feature numeriche il modello si aspetta valori numerici:")
for colonna in numeric_features:
    print(f"- {colonna}: valore numerico")


Per le feature numeriche il modello si aspetta valori numerici:
- Dependents: valore numerico
- ApplicantIncome: valore numerico
- CoapplicantIncome: valore numerico
- LoanAmount: valore numerico
- Loan_Amount_Term: valore numerico
- Credit_History: valore numerico


## 6. Dati giocattolo coerenti con il formato atteso

Costruiamo alcuni casi manuali.

Nota importante:
- i nomi delle colonne devono essere **esattamente** quelli attesi dal modello;
- le categorie testuali devono appartenere a quelle viste sopra;
- i campi numerici devono essere passati come numeri.


In [13]:
nuovi_clienti = pd.DataFrame([
    # Caso 1: profilo probabilmente favorevole
    {
        'Gender': 'Male',
        'Married': 'Yes',
        'Dependents': 0,
        'Education': 'Graduate',
        'Self_Employed': 'No',
        'ApplicantIncome': 9000,
        'CoapplicantIncome': 2500,
        'LoanAmount': 120,
        'Loan_Amount_Term': 360,
        'Credit_History': 1.0,
        'Property_Area': 'Urban'
    },
    # Caso 2: profilo probabilmente sfavorevole
    {
        'Gender': 'Female',
        'Married': 'No',
        'Dependents': 2,
        'Education': 'Not Graduate',
        'Self_Employed': 'Yes',
        'ApplicantIncome': 1800,
        'CoapplicantIncome': 0,
        'LoanAmount': 220,
        'Loan_Amount_Term': 360,
        'Credit_History': 0.0,
        'Property_Area': 'Rural'
    },
    # Caso 3: altro profilo favorevole
    {
        'Gender': 'Male',
        'Married': 'Yes',
        'Dependents': 1,
        'Education': 'Graduate',
        'Self_Employed': 'No',
        'ApplicantIncome': 6500,
        'CoapplicantIncome': 1500,
        'LoanAmount': 140,
        'Loan_Amount_Term': 360,
        'Credit_History': 1.0,
        'Property_Area': 'Semiurban'
    },
    # Caso 4: altro profilo sfavorevole
    {
        'Gender': 'Male',
        'Married': 'No',
        'Dependents': 3,
        'Education': 'Not Graduate',
        'Self_Employed': 'No',
        'ApplicantIncome': 2200,
        'CoapplicantIncome': 300,
        'LoanAmount': 250,
        'Loan_Amount_Term': 180,
        'Credit_History': 0.0,
        'Property_Area': 'Rural'
    }
])

nuovi_clienti


,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area
0,Male,Yes,0,Graduate,No,9000,2500,120,360,1.0,Urban
1,Female,No,2,Not Graduate,Yes,1800,0,220,360,0.0,Rural
2,Male,Yes,1,Graduate,No,6500,1500,140,360,1.0,Semiurban
3,Male,No,3,Not Graduate,No,2200,300,250,180,0.0,Rural


## 7. Previsioni del modello


In [14]:
predizioni = model.predict(nuovi_clienti)
probabilita = model.predict_proba(nuovi_clienti)[:, 1]

output = nuovi_clienti.copy()
output['Predizione_binaria'] = predizioni
output['Esito'] = output['Predizione_binaria'].map({1: 'APPROVATO', 0: 'RIFIUTATO'})
output['Prob_approvazione'] = np.round(probabilita, 3)

output


,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Predizione_binaria,Esito,Prob_approvazione
0,Male,Yes,0,Graduate,No,9000,2500,120,360,1.0,Urban,1,APPROVATO,0.784
1,Female,No,2,Not Graduate,Yes,1800,0,220,360,0.0,Rural,0,RIFIUTATO,0.042
2,Male,Yes,1,Graduate,No,6500,1500,140,360,1.0,Semiurban,1,APPROVATO,0.885
3,Male,No,3,Not Graduate,No,2200,300,250,180,0.0,Rural,0,RIFIUTATO,0.048


## 8. Controllo rapido: il modello restituisce sia approvati sia rifiutati?


In [15]:
output[['Esito', 'Prob_approvazione']]


,Esito,Prob_approvazione
0,APPROVATO,0.784
1,RIFIUTATO,0.042
2,APPROVATO,0.885
3,RIFIUTATO,0.048


In [17]:
print(output['Esito'].value_counts())


Esito
APPROVATO    2
RIFIUTATO    2
Name: count, dtype: int64


## 9. Funzione riutilizzabile per un singolo caso


In [18]:
def predici_mutuo(dati_cliente: dict):
    nuovo = pd.DataFrame([dati_cliente])
    classe = model.predict(nuovo)[0]
    probabilita = model.predict_proba(nuovo)[0, 1]
    return {
        'predizione_binaria': int(classe),
        'esito': 'APPROVATO' if classe == 1 else 'RIFIUTATO',
        'probabilita_approvazione': round(float(probabilita), 3)
    }


In [19]:
esempio_singolo = {
    'Gender': 'Female',
    'Married': 'Yes',
    'Dependents': 0,
    'Education': 'Graduate',
    'Self_Employed': 'No',
    'ApplicantIncome': 5200,
    'CoapplicantIncome': 1800,
    'LoanAmount': 130,
    'Loan_Amount_Term': 360,
    'Credit_History': 1.0,
    'Property_Area': 'Urban'
}

predici_mutuo(esempio_singolo)


{'predizione_binaria': 1,
 'esito': 'APPROVATO',
 'probabilita_approvazione': 0.814}

## Osservazione finale

Se in esecuzione tutti i casi risultassero approvati oppure tutti rifiutati, non sarebbe un errore del notebook: vorrebbe dire solo che questi input manuali non stanno abbastanza vicino a entrambi i lati della frontiera decisionale del modello.

In quel caso basta modificare soprattutto:
- `Credit_History`
- `ApplicantIncome`
- `LoanAmount`
- `Property_Area`

per ottenere casi più contrastati.
